# Koemi-3HIP training and runtime analysis

This notebook reports measured behavior in English. It separates legacy pre-HIP runs from the current Koemi-3HIP implementation and never treats a small smoke run as proof of Transformer-level quality.

In [ ]:
from pathlib import Path
import json
import math
import statistics
import torch
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AUTocast_DTYPE = torch.float16 if DEVICE.type == 'cuda' else torch.float32
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.facecolor': '#fffdf7',
    'figure.facecolor': 'white',
    'axes.edgecolor': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'grid.color': '#e8dfbf',
    'grid.alpha': 0.65,
})
PALETTE = {'ink': '#222222', 'gold': '#c59b2a', 'lemon': '#ead58a', 'cream': '#fff7d8', 'muted': '#8c7730'}
print({'device': str(DEVICE), 'torch_version': torch.__version__, 'autocast_dtype': str(AUTocast_DTYPE)})

## Causal validation metrics

The function below is a bounded evaluation probe. `loss` is natural-log cross-entropy, `bpb = loss / log(2)`, and answer BPB is computed only from answer-token loss. It uses `inference_mode`, limits the number of batches, and reports token denominators explicitly.

In [ ]:
from koemi.model.execution import ExecutionMode
from koemi.training.objective import calculate_training_objective, token_cross_entropy

def evaluate_limited(model, data_loader, thinking_loss_weight=1.0, maximum_batches=64):
    weighted_loss = 0.0
    weighted_thinking_loss = 0.0
    weighted_answer_loss = 0.0
    supervised_tokens = 0
    thinking_tokens = 0
    answer_tokens = 0
    token_count = 0
    expert_activations = []
    model.eval()
    autocast_enabled = DEVICE.type == 'cuda'
    with torch.inference_mode():
        for batch_index, batch in enumerate(data_loader):
            if batch_index >= maximum_batches:
                break
            supervised = int((batch['target_ids'] != -100).sum().item())
            thinking = int(((batch['target_ids'] != -100) & batch['thinking_mask']).sum().item())
            if supervised == 0:
                continue
            input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
            target_ids = batch['target_ids'].to(DEVICE, non_blocking=True)
            thinking_mask = batch['thinking_mask'].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=AUTocast_DTYPE, enabled=autocast_enabled):
                output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)
                objective = calculate_training_objective(output, target_ids, thinking_mask, thinking_loss_weight)
                token_loss = token_cross_entropy(output.logits, target_ids)
            supervised_mask = target_ids != -100
            thinking_mask = supervised_mask & thinking_mask
            answer_mask = supervised_mask & ~thinking_mask
            weighted_loss += float(objective.total_loss) * supervised
            weighted_thinking_loss += float(objective.thinking_loss) * thinking
            weighted_answer_loss += float(token_loss.masked_select(answer_mask).sum())
            supervised_tokens += supervised
            thinking_tokens += thinking
            answer_tokens += int(answer_mask.sum())
            token_count += output.token_count
            counts = output.expert_activation_counts
            if len(expert_activations) < len(counts):
                expert_activations.extend([0] * (len(counts) - len(expert_activations)))
            for index, count in enumerate(counts):
                expert_activations[index] += count
    model.train()
    result = {
        'validation_loss_nats': weighted_loss / supervised_tokens if supervised_tokens else None,
        'validation_bpb': weighted_loss / supervised_tokens / math.log(2) if supervised_tokens else None,
        'thinking_loss_nats': weighted_thinking_loss / thinking_tokens if thinking_tokens else None,
        'answer_loss_nats': weighted_answer_loss / answer_tokens if answer_tokens else None,
        'answer_bpb': weighted_answer_loss / answer_tokens / math.log(2) if answer_tokens else None,
        'supervised_tokens': supervised_tokens,
        'thinking_tokens': thinking_tokens,
        'answer_tokens': answer_tokens,
        'tokens_seen': token_count,
        'expert_activations': expert_activations,
    }
    print(json.dumps(result, indent=2, default=str))
    return result

## English training curves

Set `RUN_LOG` to a JSONL file emitted by a training run. The plot uses the fields that are present and leaves unavailable series blank. A moving median makes the trend readable without hiding raw observations.

In [ ]:
RUN_LOG = Path('artifacts/koemi-3hip-training.jsonl')
records = []
if RUN_LOG.exists():
    records = [json.loads(line) for line in RUN_LOG.read_text(encoding='utf-8').splitlines() if line.strip()]
else:
    print(f'No log found at {RUN_LOG}; run training first.')

def values(name):
    return [float(row[name]) for row in records if row.get(name) is not None]

def moving_median(series, width=7):
    if not series:
        return []
    return [statistics.median(series[max(0, i - width + 1):i + 1]) for i in range(len(series))]

fig, axes = plt.subplots(1, 2, constrained_layout=True)
if records:
    steps = list(range(1, len(records) + 1))
    loss = values('loss') or values('validation_loss')
    answer_bpb = values('answer_bpb')
    throughput = values('tokens_per_second')
    if loss:
        axes[0].plot(steps[:len(loss)], loss, color=PALETTE['lemon'], alpha=0.5, label='raw loss')
        axes[0].plot(steps[:len(loss)], moving_median(loss), color=PALETTE['gold'], linewidth=2.2, label='moving median loss')
    if answer_bpb:
        axes[0].plot(steps[:len(answer_bpb)], answer_bpb, color=PALETTE['muted'], linewidth=1.5, label='answer BPB')
    if throughput:
        axes[1].plot(steps[:len(throughput)], throughput, color=PALETTE['gold'], linewidth=2, label='tokens/s')
axes[0].set(title='Training quality', xlabel='Estimated epoch', ylabel='Value')
axes[1].set(title='Runtime throughput', xlabel='Estimated epoch', ylabel='Tokens per second')
for axis in axes:
    axis.grid(True)
    axis.legend(frameon=True)
plt.show()

## Interpretation boundary

A lower loss or higher tokens/s in one run is not a model-quality conclusion. Compare Koemi-3HIP with parameter-matched baselines, the same tokenizer, corpus, token budget, device and precision. Use at least three independent seeds for ablations, then report mean, standard deviation, p50 and p95.